# UrbanFlow AI — W3-T2 CPU training

This notebook trains the fixed W3-T2 candidates on Google Colab CPU. It preserves the repository's Python 3.11 contract, runs the focused and full test suites, trains with the configured memory guard, and downloads the resulting model artifacts.

Before opening Colab, build the upload bundle locally:

```powershell
python -m urbanflow.prepare_colab --config configs/model.json
```

Upload `artifacts/colab/urbanflow-colab-input.zip` when the next cell asks for a file. Use a CPU runtime; this pipeline does not use CUDA.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly one urbanflow-colab-input.zip bundle')
bundle_name = next(iter(uploaded))
if not bundle_name.endswith('.zip'):
    raise RuntimeError('The uploaded bundle must be a .zip file')
print(f'Uploaded {bundle_name}: {len(uploaded[bundle_name]):,} bytes')

In [ ]:
import hashlib
import json
import shutil
import zipfile
from pathlib import Path

repo = Path('/content/urbanflow-ai')
shutil.rmtree(repo, ignore_errors=True)
repo.mkdir(parents=True)
bundle = Path('/content') / bundle_name
with zipfile.ZipFile(bundle) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
        raise RuntimeError('Bundle contains an unsafe path')
    manifest = json.loads(archive.read('bundle-manifest.json'))
    archive.extractall(repo)
for item in manifest['files']:
    path = repo / item['path']
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != item['sha256']:
        raise RuntimeError(f'Checksum mismatch: {item["path"]}')
print(f"Verified {len(manifest['files'])} bundled files")

In [ ]:
import shutil
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
uv = shutil.which('uv')
venv = Path('/content/urbanflow-venv')
shutil.rmtree(venv, ignore_errors=True)
subprocess.run([uv, 'venv', '--python', '3.11', str(venv)], check=True)
python = venv / 'bin' / 'python'
subprocess.run([uv, 'pip', 'install', '--python', str(python), '-e', f'{repo}[dev]'], check=True)
subprocess.run([str(python), '--version'], check=True)

In [ ]:
focused_test = subprocess.run(
    [
        str(python),
        '-m',
        'pytest',
        'tests/test_train_model.py',
        '-vv',
        '--tb=short',
    ],
    cwd=repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print(focused_test.stdout)
if focused_test.returncode:
    raise RuntimeError(f'Focused model tests failed with exit code {focused_test.returncode}')

In [ ]:
full_test = subprocess.run(
    [str(python), '-m', 'pytest'],
    cwd=repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print(full_test.stdout)
if full_test.returncode:
    raise RuntimeError(f'Full test suite failed with exit code {full_test.returncode}')

In [ ]:
memory_check = subprocess.run(
    [
        str(python),
        '-c',
        "import psutil; m=psutil.virtual_memory(); print(f'Available RAM: {m.available / 1024**3:.2f} GiB'); raise SystemExit(0 if m.available >= 2*1024**3 else 1)",
    ],
    check=False,
)
if memory_check.returncode:
    raise RuntimeError('Colab has less than the configured 2 GiB safety floor')

In [ ]:
model_dir = repo / 'artifacts' / 'model'
model_dir.mkdir(parents=True, exist_ok=True)
train_log = model_dir / 'train.log'
with train_log.open('w', encoding='utf-8') as log:
    completed = subprocess.run(
        [str(python), '-m', 'urbanflow.train_model', '--config', 'configs/model.json'],
        cwd=repo,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
if completed.returncode:
    tail = train_log.read_text(encoding='utf-8').splitlines()[-40:]
    print('\n'.join(tail))
    raise RuntimeError(f'Training failed with exit code {completed.returncode}')
print(f'Training completed; log: {train_log}')

In [ ]:
metrics = json.loads((model_dir / 'metrics.json').read_text(encoding='utf-8'))
summary = {
    'model_version': metrics['model']['version'],
    'selected_candidate': metrics['model']['selected_candidate'],
    'boost_rounds': metrics['model']['boost_rounds'],
    'validation_mae': metrics['validation']['mae'],
    'validation_wape': metrics['validation']['wape'],
    'test_mae': metrics['test']['mae'],
    'test_wape': metrics['test']['wape'],
    'rss_peak_gib': round(metrics['resources']['rss_peak_bytes'] / 1024**3, 3),
}
summary

In [ ]:
result_base = Path('/content/urbanflow-model-artifacts')
result_zip = Path(shutil.make_archive(str(result_base), 'zip', model_dir))
print(f'Result archive: {result_zip} ({result_zip.stat().st_size:,} bytes)')
files.download(str(result_zip))